In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, approx_count_distinct, mean, lit, expr
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, StringType

def validate_data_migration(source_df, destination_df, expected_schema):
    validation_results = []

    # 1. Row Count Validation
    expected_row_count = source_df.count()
    actual_row_count = destination_df.count()
    if actual_row_count != expected_row_count:
        validation_results.append(f"Row count mismatch: Expected {expected_row_count}, Actual {actual_row_count}")

    # 2. Schema Matching
    expected_columns = set(expected_schema.get('columns', []))
    actual_columns = set(destination_df.columns)
    if not expected_columns.issubset(actual_columns):
        missing_columns = expected_columns - actual_columns
        validation_results.append(f"Missing columns in the destination: {', '.join(missing_columns)}")

    # 3. Null Value Check
    null_counts = destination_df.select([count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) for c in destination_df.columns])
    for row in null_counts.collect():
        for column, count in row.asDict().items():
            if count > 0:
                validation_results.append(f"Column '{column}' has {count} NULL values after migration.")

    # 4. Data Type Validation
    expected_data_types = expected_schema.get('data_types', {})
    actual_schema = {field.name: field.dataType for field in destination_df.schema.fields}
    for column, expected_dtype in expected_data_types.items():
        if column in actual_schema and actual_schema[column] != expected_dtype:
            validation_results.append(f"Column '{column}' has unexpected data type {actual_schema[column]}. Expected {expected_dtype}.")

    # 5. Unique Value Check
    unique_counts = destination_df.select([approx_count_distinct(col(c)).alias(c) for c in destination_df.columns])
    for row in unique_counts.collect():
        for column, count in row.asDict().items():
            if count == destination_df.count():
                validation_results.append(f"Column '{column}' has all unique values.")

    # 6. Data Consistency Check (Example: Binary columns should have only 0s and 1s)
    for column in expected_columns:
        if column in destination_df.columns:
            unique_values = destination_df.select(col(column)).distinct().rdd.flatMap(lambda x: x).collect()
            if len(unique_values) > 2:
                validation_results.append(f"Column '{column}' has more than 2 unique values, inconsistent with expected data.")

    # 7. Data Distribution Check (Example: Ensure categorical column has expected diversity)
    for column in expected_columns:
        if column in destination_df.columns and destination_df.schema[column].dataType == StringType():
            value_counts = destination_df.groupBy(column).count().count()
            if value_counts < 2:
                validation_results.append(f"Column '{column}' has less than 2 unique values, inconsistent with expected data distribution.")

    # 8. Outlier Detection (Using IQR method)
    for column in expected_columns:
        if column in destination_df.columns and destination_df.schema[column].dataType in [IntegerType(), FloatType()]:
            quantiles = destination_df.approxQuantile(column, [0.25, 0.75], 0.05)
            if len(quantiles) == 2:
                q1, q3 = quantiles
                iqr = q3 - q1
                lower_bound = q1 - 1.5 * iqr
                upper_bound = q3 + 1.5 * iqr
                outliers = destination_df.filter((col(column) < lower_bound) | (col(column) > upper_bound))
                if outliers.count() > 0:
                    validation_results.append(f"Column '{column}' has outliers.")

    # 9. Accuracy Check (Compare with Reference Data)
    reference_df = expected_schema.get('reference_data')
    if reference_df is not None:
        for column in expected_columns:
            if column in destination_df.columns:
                diff_count = destination_df.select(col(column)).subtract(reference_df.select(col(column))).count()
                if diff_count > 0:
                    validation_results.append(f"Column '{column}' has {diff_count} mismatched records compared to reference data.")

    # 10. Missing Row Detection
    missing_rows_df = expected_schema.get('missing_rows')
    if missing_rows_df is not None:
        missing_rows_count = missing_rows_df.count()
        if missing_rows_count > 0:
            validation_results.append(f"Missing rows detected in the destination: {missing_rows_count}")

    # 11. Metadata Validation (Basic Check)
    for column in expected_columns:
        if column in destination_df.columns and destination_df.schema[column].dataType == StringType():
            metadata_check = destination_df.select(col(column)).distinct().rdd.flatMap(lambda x: x).collect()
            if metadata_check != expected_schema.get('metadata', {}).get(column, []):
                validation_results.append(f"Metadata mismatch for column '{column}'.")

    # 12. Regression Testing (Compare mean values before and after migration)
    numerical_columns = [f.name for f in destination_df.schema.fields if isinstance(f.dataType, (IntegerType, FloatType))]
    for column in numerical_columns:
        if column in source_df.columns:
            source_mean = source_df.select(mean(col(column))).collect()[0][0]
            destination_mean = destination_df.select(mean(col(column))).collect()[0][0]
            if source_mean != destination_mean:
                validation_results.append(f"Mean value mismatch for column '{column}': Source {source_mean}, Destination {destination_mean}")

    return validation_results

# Sample Schema Definition
expected_schema = {
    'columns': ['City', 'AvgTemperature'],
    'data_types': {'City': StringType(), 'AvgTemperature': FloatType()},
    'missing_rows': None,  # Placeholder if needed
    'reference_data': None  # Placeholder for accuracy validation
}

# Load Data in PySpark
source_df = spark.read.csv("dbfs:/FileStore/tables/city_temperature.csv", header=True, inferSchema=True)
destination_df = spark.read.csv("dbfs:/FileStore/tables/city_temperature.csv", header=True, inferSchema=True)

# Run Validation
validation_results = validate_data_migration(source_df, destination_df, expected_schema)

# Print Results
for result in validation_results:
    print(result)
